In [1]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import time
import logging

In [15]:
# Configurar logging para capturar erros detalhados
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Cliente otimizado para vLLM (muito mais rápido que LM Studio)
client = OpenAI(
    base_url="http://localhost:1234/v1", 
    api_key="vllm",  # Muda para vLLM
    timeout=120.0,   # Reduzido para 2 min (vLLM é 5x mais rápido)
    max_retries=2    # Menos retries necessárias
)
MODELO_GEN = "Qwen/Qwen3-8B-AWQ"  # Nome exato do modelo rodando em vLLM

def limpar_thinking(texto):
    """Remove tags de thinking/reasoning do relatório."""
    import re
    # Remove <think>...</think>
    texto = re.sub(r'<think>.*?</think>', '', texto, flags=re.DOTALL)
    # Remove <reasoning>...</reasoning>
    texto = re.sub(r'<reasoning>.*?</reasoning>', '', texto, flags=re.DOTALL)
    # Remove <thought>...</thought>
    texto = re.sub(r'<thought>.*?</thought>', '', texto, flags=re.DOTALL)
    # Remove linhas com "Thought:", "Reasoning:", etc
    linhas = [l for l in texto.split('\n') if not l.strip().startswith(('Thought:', 'Reasoning:', 'Think:'))]
    return '\n'.join(linhas).strip()

def gerar_relatorios_em_lote(csv_entrada, n_amostras=100, delay_entre_requisicoes=0.5):
    """
    Gera relatórios técnicos com retry robusto e delays.
    Args:
        csv_entrada: arquivo CSV de entrada
        n_amostras: número de amostras a processar
        delay_entre_requisicoes: delay em segundos entre requisições (otimizado para vLLM)
    """
    df = pd.read_csv(csv_entrada, sep=',', on_bad_lines='skip')
    df_sample = df.sample(n=min(n_amostras, len(df)), random_state=42)
    
    lista_relatorios = []
    erros_ocorridos = []
    contador_sucesso = 0
    contador_erro = 0

    print(f"Gerando relatórios para {len(df_sample)} registros com vLLM...")
    print(f"Modelo: {MODELO_GEN} | Max context: 16k tokens\n")

    for idx, (_, row) in enumerate(tqdm(df_sample.iterrows(), total=len(df_sample))):
        # Preparação do Contexto Extraído do seu Dataset
        contexto_fato = (
            f"CODIGO: {row.get('CODIGO', '')} | "
            f"NATUREZA: {row.get('NATUREZA_OCORRENCIA', '')} | "
            f"DATA: {row.get('DATA_FATO', '')} | "
            f"HORA: {row.get('HORA_FATO', '')} | "
            f"LOCAL: {row.get('MUNICIPIO', '')}/{row.get('BAIRRO', '')} | "
            f"TIPO LOCAL: {row.get('TIPO_LOCAL_FATO', '')} | "
            f"NARRATIVA: {row.get('NARRATIVA', '')}"
        )
        
        prompt_especifico = (
            f"Escreva um RELATÓRIO TÉCNICO RESUMIDO baseado nos dados abaixo:\n{contexto_fato}\n\n"
            "Siga rigorosamente este formato:\n"
            "**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCORRÊNCIA**\n"
            "**DATA DOS FATOS:**...\n"
            "**HORA INICIAL DOS FATOS:**...\n"
            "**LOCAL:**...\n"
            "**NATUREZA DA OCORRÊNCIA:**...\n"
            "**RELATO SUCINTO DOS ACONTECIMENTOS**:..."
        )

        # Retry com backoff exponencial
        max_tentativas = 3
        tentativa = 0
        sucesso = False
        
        while tentativa < max_tentativas and not sucesso:
            try:
                # Delay para não sobrecarregar vLLM
                if tentativa > 0:
                    tempo_espera = (2 ** tentativa)  # Backoff exponencial: 2s, 4s, 8s
                    logger.info(f"Aguardando {tempo_espera}s antes de retry #{tentativa}...")
                    time.sleep(tempo_espera)
                else:
                    time.sleep(delay_entre_requisicoes)
                
                res = client.chat.completions.create(
                    model=MODELO_GEN,
                    messages=[{"role": "user", "content": prompt_especifico}],
                    temperature=0.6,
                    stop=["Thought:", "Reasoning:"]
                )
                relatorio = res.choices[0].message.content
                relatorio = limpar_thinking(relatorio)  # Remove tags de thinking
                
                lista_relatorios.append({
                    "codigo_bo": row.get('CODIGO', ''),
                    "narrativa_original": row.get('NARRATIVA', ''),
                    "contexto_completo": contexto_fato,
                    "relatorio_ia": relatorio
                })
                contador_sucesso += 1
                sucesso = True
                
            except Exception as e:
                tentativa += 1
                codigo_bo = row.get('CODIGO', 'DESCONHECIDO')
                tipo_erro = type(e).__name__
                
                if tentativa < max_tentativas:
                    logger.warning(f"Tentativa {tentativa}/{max_tentativas} falhou para BO {codigo_bo}: {tipo_erro}. Retentando...")
                else:
                    contador_erro += 1
                    mensagem_erro = f"Erro no BO {codigo_bo} (item {idx+1}): {tipo_erro} - {str(e)[:200]}"
                    print(f"\n❌ {mensagem_erro}")
                    logger.error(mensagem_erro)
                    erros_ocorridos.append({
                        "indice": idx + 1,
                        "codigo_bo": codigo_bo,
                        "tipo_erro": tipo_erro,
                        "mensagem": str(e)[:500]
                    })

    df_final = pd.DataFrame(lista_relatorios)
    df_final.to_csv("dataset_com_relatorios.csv", index=False, encoding='utf-8')
    
    # Salvar relatório de erros
    if erros_ocorridos:
        df_erros = pd.DataFrame(erros_ocorridos)
        df_erros.to_csv("erros_geracao_relatorios.csv", index=False, encoding='utf-8')
    
    print(f"\n{'='*60}")
    print(f"Etapa de Geração Concluída.")
    print(f"✅ Sucessos: {contador_sucesso}/{len(df_sample)}")
    print(f"❌ Erros: {contador_erro}/{len(df_sample)}")
    print(f"Taxa de sucesso: {(contador_sucesso/len(df_sample)*100):.1f}%")
    print(f"{'='*60}")

In [16]:
gerar_relatorios_em_lote("../Dataset/dataset_tratado.csv", n_amostras=200, delay_entre_requisicoes=5)

Gerando relatórios para 200 registros com vLLM...
Modelo: Qwen/Qwen3-8B-AWQ | Max context: 16k tokens



  0%|          | 0/200 [00:00<?, ?it/s]2026-03-22 23:32:46,793 - DEBUG - Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-03a4333d-9471-440e-988d-c2aa1758bef5', 'content': None, 'json_data': {'messages': [{'role': 'user', 'content': 'Escreva um RELATÓRIO TÉCNICO RESUMIDO baseado nos dados abaixo:\nCODIGO: 1997 | NATUREZA: OUTRAS FRAUDES | DATA: 2025-03-15 00:00:00.000 | HORA: 1970-01-01 16:35:00 | LOCAL: CUIABA/JARDIM MARIANA | TIPO LOCAL: INTERNET | NARRATIVA: comunicante começou receber ligações mensagens      pessoas perguntando agiota  realiza emprestimos valores  comunicante negou perguntava pessoas aonde indicou ele  pessoas vendo anuncio facebook comentario  marketplace  onde consta nome telefone  diante desses contatos recebidos preocupado alguma fraude relacionado nome pede facebook retire imediatamente anuncios  registrase \n\nSiga rigorosamente este formato:\n**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OC


Etapa de Geração Concluída.
✅ Sucessos: 200/200
❌ Erros: 0/200
Taxa de sucesso: 100.0%
